# MemMachine Memory Integration with NeMo Agent Toolkit

This notebook demonstrates how to use MemMachine memory inside NeMo Agent Toolkit end-to-end. MemMachine provides a unified memory management system that supports both episodic (conversation-based) and semantic (fact/preference-based) memory through a single interface.

## What You'll Learn

- How to set up and configure MemMachine server
- How to integrate MemMachine memory with NeMo Agent Toolkit
- How to add and retrieve episodic memories (conversation-based)
- How to add and retrieve semantic memories (fact/preference-based)
- How to use memory in an agent workflow with tools

## Table of Contents

- [0) Setup](#setup)
  - [0.1) Prerequisites](#prereqs)
  - [0.2) MemMachine Server Setup](#memmachine-setup)
  - [0.3) API Keys](#api-keys)
  - [0.4) Installing Dependencies](#installing-deps)
  - [0.5) Verify MemMachine Server](#verify-server)
- [1) Basic Memory Operations](#basic-memory)
  - [1.1) Programmatic Memory Usage](#programmatic)
  - [1.2) Adding Episodic Memories](#episodic)
  - [1.3) Adding Semantic Memories](#semantic)
  - [1.4) Searching Memories](#searching)
- [2) Agent Workflow with Memory](#agent-workflow)
  - [2.1) Create Configuration](#config)
  - [2.2) Run Agent with Memory](#run-agent)
- [3) Next Steps](#next-steps)

<span style="color:rgb(0, 31, 153); font-style: italic;">Note: In Google Colab use the Table of Contents tab to navigate.</span>

<a id="setup"></a>
## 0) Setup

<a id="prereqs"></a>
### 0.1) Prerequisites

- **Platform:** Linux, macOS, or Windows
- **Python:** version 3.11, 3.12, or 3.13
- **MemMachine Server:** Must be installed and running (see next section)
- **Neo4j Database:** Required by MemMachine (can be auto-installed)
- **LLM Provider:** OpenAI, AWS Bedrock, or Ollama (for semantic memory processing)

<a id="memmachine-setup"></a>
### 0.2) MemMachine Server Setup

**IMPORTANT:** Before running this notebook, you must set up and start the MemMachine server.

#### Step 1: Install MemMachine Server
```bash
pip install memmachine-server
```

#### Step 2: Run Configuration Wizard
```bash
memmachine-configure
```

The wizard will guide you through:
- **Neo4j Database**: Option to install Neo4j automatically or provide connection details
- **LLM Provider**: Choose from OpenAI, AWS Bedrock, or Ollama
- **Model Selection**: Select LLM and embedding models
- **API Keys**: Input necessary credentials
- **Server Settings**: Configure host and port (default: localhost:8080)

Configuration file will be saved at: `~/.config/memmachine/cfg.yml`

#### Step 3: Start MemMachine Server
In a separate terminal, run:
```bash
memmachine-server
```

The server will start on `http://localhost:8080` by default.

For more details, see the [MemMachine Documentation](https://docs.memmachine.ai/open_source/configuration-wizard).

<a id="api-keys"></a>
### 0.3) API Keys

For this notebook, you will need the following API keys:

- **NVIDIA Build:** You can obtain an NVIDIA Build API Key by creating an [NVIDIA Build](https://build.nvidia.com) account and generating a key at https://build.nvidia.com/settings/api-keys

**Note:** MemMachine's LLM provider API keys (e.g., OpenAI) are configured in the MemMachine `cfg.yml` file, not here.

In [1]:
# Verify that the langchain plugin is properly installed and registered
import sys
from importlib.metadata import entry_points

try:
    # Check if the entry point exists
    components = entry_points(group='nat.components')
    langchain_entry = None
    for ep in components:
        if 'langchain' in ep.name.lower() and 'tools' not in ep.name.lower():
            langchain_entry = ep
            break
    
    if langchain_entry:
        print(f"Found langchain entry point: {langchain_entry.name} -> {langchain_entry.module}")
        
        # Check if the module path is correct
        expected_module = "nat.plugins.langchain.register"
        if langchain_entry.module != expected_module:
            print(f"❌ Entry point module path is incorrect!")
            print(f"   Expected: {expected_module}")
            print(f"   Found:    {langchain_entry.module}")
        else:
            print(f"✅ Entry point module path is correct: {langchain_entry.module}")
            try:
                # Try to load the module to verify it works
                langchain_entry.load()
                print("✅ LangChain plugin module loaded successfully!")
            except Exception as e:
                print(f"❌ Failed to load langchain plugin module: {e}")
                print("\n💡 Try reinstalling the package:")
                print("   uv pip install --force-reinstall \"nvidia-nat[langchain]\"")
    else:
        print("❌ langchain entry point not found!")
        print("\n💡 The langchain plugin may not be installed correctly.")
        print("   Try reinstalling:")
        print("   uv pip install --force-reinstall \"nvidia-nat[langchain]\"")
        print("\n   Or if you're in the repo directory:")
        print("   uv pip install --force-reinstall -e packages/nvidia_nat_langchain")
except Exception as e:
    print(f"❌ Error checking entry points: {e}")
    print("\n💡 Try reinstalling the package:")
    print("   uv pip install --force-reinstall \"nvidia-nat[langchain]\"")

Found langchain entry point: nat_langchain -> nat.plugins.langchain.register
✅ Entry point module path is correct: nat.plugins.langchain.register
✅ LangChain plugin module loaded successfully!


/Users/charlieyi/Desktop/nemo-agent-toolkit/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

In [3]:
# Verify that the memmachine plugin is properly installed and registered
import sys
from importlib.metadata import entry_points

try:
    # Check if the entry point exists
    components = entry_points(group='nat.components')
    memmachine_entry = None
    for ep in components:
        if 'memmachine' in ep.name.lower():
            memmachine_entry = ep
            break
    
    if memmachine_entry:
        print(f"Found memmachine entry point: {memmachine_entry.name} -> {memmachine_entry.module}")
        
        # Check if the module path is correct
        expected_module = "nat.plugins.memmachine.register"
        if memmachine_entry.module != expected_module:
            print(f"❌ Entry point module path is incorrect!")
            print(f"   Expected: {expected_module}")
            print(f"   Found:    {memmachine_entry.module}")
            print("\n💡 The package needs to be reinstalled in editable mode.")
            print("   Run this command in a terminal:")
            print("   uv pip install --force-reinstall -e ../../packages/memmachine")
            print("\n   Or if you're in the repo root:")
            print("   uv pip install --force-reinstall -e packages/memmachine")
        else:
            print(f"✅ Entry point module path is correct: {memmachine_entry.module}")
            try:
                # Try to load the module to verify it works
                memmachine_entry.load()
                print("✅ Plugin module loaded successfully!")
            except Exception as e:
                print(f"❌ Failed to load plugin module: {e}")
                print("\n💡 Try reinstalling the package in editable mode:")
                print("   uv pip install --force-reinstall -e ../../packages/memmachine")
    else:
        print("❌ memmachine entry point not found!")
        print("\n💡 The plugin may not be installed correctly.")
        print("   Try reinstalling in editable mode:")
        print("   uv pip install --force-reinstall -e ../../packages/memmachine")
        print("\n   Or if you're not in the repo directory:")
        print("   uv pip install --force-reinstall nvidia-nat-memmachine")
except Exception as e:
    print(f"❌ Error checking entry points: {e}")
    print("\n💡 Try reinstalling the package:")
    print("   uv pip install --force-reinstall -e ../../packages/memmachine")

Found memmachine entry point: nat_memmachine -> nat.plugins.memmachine.register
✅ Entry point module path is correct: nat.plugins.memmachine.register
✅ Plugin module loaded successfully!


<a id="installing-deps"></a>
### 0.4) Installing Dependencies

The recommended way to install NAT is through `pip` or `uv pip`.

First, we will install `uv` which offers parallel downloads and faster dependency resolution.

In [4]:
!pip install uv

Now install NeMo Agent Toolkit with langchain support and the MemMachine integration:

In [5]:
%%bash
# Install nvidia-nat with langchain support
# This is REQUIRED for react_agent workflow type
uv pip show -q "nvidia-nat-langchain"
if [ $? -ne 0 ]; then
    echo "Installing nvidia-nat[langchain]..."
    uv pip install "nvidia-nat[langchain]"
    echo "✅ Installation complete"
else
    echo "nvidia-nat[langchain] is already installed"
    echo ""
    echo "If you encounter 'No module named nat.plugins.langchain' errors,"
    echo "try reinstalling:"
    echo "  uv pip install --force-reinstall 'nvidia-nat[langchain]'"
fi

nvidia-nat[langchain] is already installed

If you encounter 'No module named nat.plugins.langchain' errors,
try reinstalling:
  uv pip install --force-reinstall 'nvidia-nat[langchain]'


In [6]:
%%bash
# Install nvidia-nat-memmachine package
# IMPORTANT: For the plugin to work with YAML configs, it must be installed
# in editable mode if developing locally, or entry points won't be registered
uv pip show -q "nvidia-nat-memmachine"
if [ $? -ne 0 ]; then
    # Try installing from local source first (for development)
    if [ -d "../../packages/memmachine" ]; then
        echo "Installing nvidia-nat-memmachine in editable mode from local source..."
        uv pip install -e ../../packages/memmachine
    else
        echo "Installing nvidia-nat-memmachine from PyPI..."
        uv pip install "nvidia-nat-memmachine"
    fi
else
    echo "nvidia-nat-memmachine is already installed"
    echo ""
    echo "If you encounter plugin loading errors, try reinstalling:"
    echo "  uv pip install --force-reinstall -e ../../packages/memmachine"
fi

nvidia-nat-memmachine is already installed

If you encounter plugin loading errors, try reinstalling:
  uv pip install --force-reinstall -e ../../packages/memmachine


In [7]:
%%bash
uv pip show -q "memmachine-server"
if [ $? -ne 0 ]; then
    uv pip install "memmachine-server~=0.2.2"
else
    echo "memmachine is already installed"
fi

memmachine is already installed


<a id="verify-server"></a>
### 0.5) Verify MemMachine Server is Running

Let's check if the MemMachine server is accessible:

In [8]:
import requests

# MemMachine server URL (default)
MEMMACHINE_BASE_URL = os.environ.get("MEMMACHINE_BASE_URL", "http://localhost:8080")

try:
    response = requests.get(f"{MEMMACHINE_BASE_URL}/api/v2/health", timeout=5)
    if response.status_code == 200:
        print(f"✅ MemMachine server is running at {MEMMACHINE_BASE_URL}")
    else:
        print(f"⚠️  MemMachine server responded with status {response.status_code}")
except requests.exceptions.ConnectionError:
    print(f"❌ Cannot connect to MemMachine server at {MEMMACHINE_BASE_URL}")
    print("Please ensure the server is running. Start it with: memmachine-server")
except Exception as e:
    print(f"❌ Error checking server: {e}")

✅ MemMachine server is running at http://localhost:8080


<a id="basic-memory"></a>
## 1) Basic Memory Operations

Let's explore how to use MemMachine memory programmatically with NeMo Agent Toolkit.

<a id="programmatic"></a>
### 1.1) Programmatic Memory Usage

First, let's import the necessary modules:

In [9]:
import asyncio
import uuid
from nat.builder.workflow_builder import WorkflowBuilder
from nat.data_models.config import GeneralConfig
from nat.memory.models import MemoryItem
from nat.plugins.memmachine.memory import MemMachineMemoryClientConfig

# Create a unique test ID for this session
test_id = str(uuid.uuid4())[:8]
print(f"Test session ID: {test_id}")

Test session ID: f888f411


Now, let's configure and create a MemMachine memory client:

In [10]:
# Configure MemMachine memory client
memmachine_config = MemMachineMemoryClientConfig(
    base_url=MEMMACHINE_BASE_URL,
    org_id=f"demo_org_{test_id}",
    project_id=f"demo_project_{test_id}",
    timeout=30,
    max_retries=3
)

print(f"✅ MemMachine configuration created")
print(f"   Base URL: {memmachine_config.base_url}")
print(f"   Org ID: {memmachine_config.org_id}")
print(f"   Project ID: {memmachine_config.project_id}")

✅ MemMachine configuration created
   Base URL: http://localhost:8080
   Org ID: demo_org_f888f411
   Project ID: demo_project_f888f411


<a id="episodic"></a>
### 1.2) Adding Episodic Memories

Episodic memories are conversation-based and preserve the full context of interactions. Let's add an episodic memory:

In [11]:
async def add_episodic_memory():
    """Add an episodic memory (conversation-based)"""
    general_config = GeneralConfig()
    
    async with WorkflowBuilder(general_config=general_config) as builder:
        # Add MemMachine memory client
        await builder.add_memory_client("memmachine_memory", memmachine_config)
        memory_client = await builder.get_memory_client("memmachine_memory")
        
        # Create an episodic memory with conversation
        user_id = f"demo_user_{test_id}"
        conversation = [
            {"role": "user", "content": "I love pizza and Italian food, especially margherita pizza."},
            {"role": "assistant", "content": "I'll remember that you love pizza and Italian food, especially margherita pizza."},
        ]
        
        memory_item = MemoryItem(
            conversation=conversation,
            user_id=user_id,
            memory="User loves pizza and Italian food, especially margherita pizza",
            metadata={
                "session_id": f"session_{test_id}",
                "agent_id": f"agent_{test_id}",
                "test_id": "episodic_demo"
            },
            tags=["food", "preference", "italian"]
        )
        
        # Add the memory
        await memory_client.add_items([memory_item])
        print(f"✅ Added episodic memory for user: {user_id}")
        
        # Wait a moment for indexing
        await asyncio.sleep(2)
        
        return user_id, memory_client

# Run the async function
user_id, memory_client = await add_episodic_memory()

Failed to get project demo_org_f888f411/demo_project_f888f411
Traceback (most recent call last):
  File "/Users/charlieyi/Desktop/nemo-agent-toolkit/venv/lib/python3.12/site-packages/memmachine/rest_client/client.py", line 282, in get_project
    response.raise_for_status()
  File "/Users/charlieyi/Desktop/nemo-agent-toolkit/venv/lib/python3.12/site-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 404 Client Error: Not Found for url: http://localhost:8080/api/v2/projects/get


✅ Added episodic memory for user: demo_user_f888f411


<a id="semantic"></a>
### 1.3) Adding Semantic Memories

Semantic memories are fact/preference-based and are processed asynchronously. They're great for storing long-term user preferences and facts:

In [12]:
async def add_semantic_memory():
    """Add a semantic memory (fact/preference-based) using the existing memory client"""
    # Reuse the memory_client from the episodic memory cell
    # This avoids trying to create the project again
    
    # Create a semantic memory (fact/preference)
    semantic_memory = MemoryItem(
        conversation=None,  # No conversation for semantic memory
        user_id=user_id,
        memory="User prefers working in the morning (9 AM - 12 PM) and is allergic to peanuts",
        metadata={
            "session_id": f"session_{test_id}",
            "agent_id": f"agent_{test_id}",
            "use_semantic_memory": True,  # Explicitly mark as semantic
            "test_id": "semantic_demo"
        },
        tags=["preference", "allergy", "schedule"]
    )
    
    # Add the memory using the existing memory_client
    await memory_client.add_items([semantic_memory])
    print(f"✅ Added semantic memory for user: {user_id}")
    
    # Semantic memories are processed asynchronously
    # Wait longer for background ingestion task
    print("⏳ Waiting for semantic memory ingestion (this may take 2-5 seconds)...")
    await asyncio.sleep(5)
    
    return memory_client

memory_client = await add_semantic_memory()

✅ Added semantic memory for user: demo_user_f888f411
⏳ Waiting for semantic memory ingestion (this may take 2-5 seconds)...


<a id="searching"></a>
### 1.4) Searching Memories

Now let's search for the memories we just added. **Note:** MemMachine's search function returns results from both episodic and semantic memory in a single search call, combining them into one list.

In [13]:
async def search_memories():
    """Search for memories - returns both episodic and semantic in one call"""
    # Reuse the memory_client from the episodic memory cell
    # This avoids trying to create the project again
    
    # Single search returns both episodic and semantic memories
    print("🔍 Searching for memories (pizza/Italian food)...")
    print("   Note: This search returns both episodic and semantic memories combined\n")
    
    all_results = await memory_client.search(
        query="pizza Italian food margherita",
        top_k=10,
        user_id=user_id,
        session_id=f"session_{test_id}",
        agent_id=f"agent_{test_id}"
    )
    
    print(f"   Found {len(all_results)} total memories (episodic + semantic combined)\n")
    
    # Display results - episodic memories typically have conversation, semantic don't
    for i, mem in enumerate(all_results, 1):
        memory_type = "Episodic" if mem.conversation else "Semantic"
        print(f"   {i}. [{memory_type}] {mem.memory}")
        if mem.conversation:
            print(f"      Conversation: {len(mem.conversation)} messages")
        if mem.tags:
            print(f"      Tags: {', '.join(mem.tags)}")
        print()
    
    # Now search for semantic memory (may need retries due to async processing)
    print("\n🔍 Searching for semantic memory (morning work allergy)...")
    print("   Note: Semantic memories may take a few seconds to be searchable\n")
    
    semantic_results = []
    for attempt in range(3):
        semantic_results = await memory_client.search(
            query="morning work schedule allergy peanuts",
            top_k=10,
            user_id=user_id,
            session_id=f"session_{test_id}",
            agent_id=f"agent_{test_id}"
        )
        # Filter for semantic memories (no conversation)
        semantic_only = [m for m in semantic_results if not m.conversation]
        if len(semantic_only) > 0:
            semantic_results = semantic_only
            break
        print(f"   Attempt {attempt + 1}: No semantic results yet, waiting...")
        await asyncio.sleep(2)
    
    print(f"   Found {len(semantic_results)} semantic memories")
    for i, mem in enumerate(semantic_results, 1):
        print(f"   {i}. {mem.memory}")
        if mem.tags:
            print(f"      Tags: {', '.join(mem.tags)}")

await search_memories()

🔍 Searching for memories (pizza/Italian food)...
   Note: This search returns both episodic and semantic memories combined

   Found 1 total memories (episodic + semantic combined)

   1. [Episodic] I'll remember that you love pizza and Italian food, especially margherita pizza.
      Conversation: 2 messages
      Tags: food, preference, italian


🔍 Searching for semantic memory (morning work allergy)...
   Note: Semantic memories may take a few seconds to be searchable

   Attempt 1: No semantic results yet, waiting...
   Attempt 2: No semantic results yet, waiting...
   Attempt 3: No semantic results yet, waiting...
   Found 1 semantic memories
   1. I'll remember that you love pizza and Italian food, especially margherita pizza.
      Tags: food, preference, italian


<a id="agent-workflow"></a>
## 2) Agent Workflow with Memory

Now let's create an agent workflow that can use memory tools to remember and recall information.

<a id="config"></a>
### 2.1) Create Configuration

Let's create a YAML configuration file for an agent with memory capabilities:

**Note:** The `react_agent` workflow type requires `nvidia-nat[langchain]` to be installed. This is because the ReAct agent uses LangChain/LangGraph for its agent framework. If you encounter errors about missing `langchain` modules (like `No module named 'langchain.schema'`), make sure you've run the installation cell above that installs `nvidia-nat[langchain]`. The ReAct agent is one of several agent types available in NeMo Agent Toolkit - it requires langchain because it uses LangGraph for agent orchestration.

In [14]:
# Write the agent config file using the same test_id from Part 1
# This ensures the agent uses the same MemMachine project we created earlier

agent_config = f'''general:
  telemetry:
    enabled: false

llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.7
    max_tokens: 1024

memory:
  memmachine_memory:
    _type: memmachine_memory
    base_url: "http://localhost:8080"
    org_id: "{memmachine_config.org_id}"
    project_id: "{memmachine_config.project_id}"

functions:
  get_memory:
    _type: get_memory
    memory: memmachine_memory
    description: |
      Retrieve memories relevant to a query. Always call this tool first to check for existing user preferences or facts.
      Use the exact JSON format with user_id, query, and top_k parameters.

  add_memory:
    _type: add_memory
    memory: memmachine_memory
    description: |
      Add facts about user preferences or information to long-term memory.
      Use the exact JSON format with user_id, memory, conversation (optional), metadata, and tags.

workflow:
  _type: react_agent
  tool_names: [add_memory, get_memory]
  description: "A chat agent that can remember and recall user preferences using MemMachine memory"
  llm_name: nim_llm
  verbose: true
  system_prompt: |
    Answer the following questions as best you can. You may ask the human to use the following tools:

    {{tools}}

    IMPORTANT MEMORY TOOL REQUIREMENTS:
    1. You MUST use get_memory tool FIRST, before calling any other tools, to check for existing user preferences
    2. You MUST use user_id "{user_id}" for all memory operations
    3. You MUST include ALL required parameters when calling memory tools
    4. When calling add_memory or get_memory, you MUST use the exact format as below, don't include any other content,
      and make sure the input is a valid JSON object.

    For get_memory tool, you MUST use this exact format:
    {{{{
        "query": "user preferences",
        "top_k": 5,
        "user_id": "{user_id}"
    }}}}

    For add_memory tool, you MUST use this exact format:
    {{{{
        "conversation": [
            {{{{
                "role": "user",
                "content": "I prefer chocolate ice cream"
            }}}},
            {{{{
                "role": "assistant",
                "content": "I'll remember that you prefer chocolate ice cream."
            }}}}
        ],
        "user_id": "{user_id}",
        "metadata": {{{{
            "key_value_pairs": {{{{
                "type": "preference",
                "category": "food"
            }}}}
        }}}},
        "memory": "User prefers chocolate ice cream.",
        "tags": ["food", "preference", "ice-cream"]
    }}}}

    You may respond in one of two formats.
    Use the following format exactly to ask the human to use a tool:

    Question: the input question you must answer
    Thought: you should always think about what to do
    Action: the action to take, should be one of [{{tool_names}}]
    Action Input: the input to the action (if there is no required input, include "Action Input: None")
    Observation: wait for the human to respond with the result from the tool, do not assume the response

    ... (this Thought/Action/Action Input/Observation can repeat N times)
    Use the following format once you have the final answer:

    Thought: I now know the final answer
    Final Answer: the final answer to the original input question
'''

# Write the config file
with open('memmachine_agent_config.yml', 'w') as f:
    f.write(agent_config)

print(f"✅ Created agent config with project: {memmachine_config.org_id}/{memmachine_config.project_id}")
print(f"   Config file: memmachine_agent_config.yml")

✅ Created agent config with project: demo_org_f888f411/demo_project_f888f411
   Config file: memmachine_agent_config.yml


In [15]:
# Verify that langchain plugin can be imported in the current Python environment
import sys
print(f"Current Python: {sys.executable}")
print(f"Python version: {sys.version}\n")

try:
    # Try to import the langchain plugin module directly
    from nat.plugins.langchain import register
    print("✅ Successfully imported nat.plugins.langchain.register")
    print("✅ LangChain plugin is available in this Python environment\n")
    
    # Also verify entry points
    from importlib.metadata import entry_points
    components = entry_points(group='nat.components')
    langchain_eps = [ep for ep in components if 'langchain' in ep.name.lower() and 'tools' not in ep.name.lower()]
    
    if langchain_eps:
        print(f"✅ Found langchain entry point: {langchain_eps[0].name}")
        print("✅ Ready to run agent workflows with langchain!\n")
    else:
        print("⚠️  Warning: LangChain entry point not found, but module can be imported")
        print("   This may still work, but entry points should be registered.\n")
        
except ImportError as e:
    print(f"❌ Failed to import langchain plugin: {e}")
    print("\n💡 The langchain plugin is not available in this Python environment.")
    print("   This means `!nat run` will fail when trying to use react_agent workflow.")
    print("\n   Solutions:")
    print("   1. Make sure you've run the installation cell above that installs nvidia-nat[langchain]")
    print("   2. Restart the Jupyter kernel after installing")
    print("   3. Verify you're using the correct Python environment (venv)")
    print(f"      Current Python: {sys.executable}")
    print("\n   If the issue persists, try reinstalling:")
    print("      uv pip install --force-reinstall 'nvidia-nat[langchain]'")
    raise

Current Python: /Users/charlieyi/Desktop/nemo-agent-toolkit/venv/bin/python
Python version: 3.12.9 (v3.12.9:fdb81425a9a, Feb  4 2025, 12:21:36) [Clang 13.0.0 (clang-1300.0.29.30)]

✅ Successfully imported nat.plugins.langchain.register
✅ LangChain plugin is available in this Python environment

✅ Found langchain entry point: nat_langchain
✅ Ready to run agent workflows with langchain!



<a id="run-agent"></a>
### 2.2) Run Agent with Memory

First, let's add an initial memory that the agent can recall:

In [16]:
# Add a memory about user preferences using the SAME project from Part 1
# This reuses the memmachine_config created earlier, ensuring consistency
async def setup_initial_memory():
    """Add an initial memory that the agent can recall - uses existing project from Part 1"""
    general_config = GeneralConfig()
    
    # Reuse the memmachine_config from Part 1 - no need to create a new config!
    # The project was already created in Step 1.1
    print(f"Using existing project: {memmachine_config.org_id}/{memmachine_config.project_id}")
    
    async with WorkflowBuilder(general_config=general_config) as builder:
        await builder.add_memory_client("memmachine_memory", memmachine_config)
        memory_client = await builder.get_memory_client("memmachine_memory")
        
        # Add a memory about user preferences
        # Use the same user_id as in Part 1 for consistency
        initial_memory = MemoryItem(
            conversation=[
                {"role": "user", "content": "My favorite programming language is Python"},
                {"role": "assistant", "content": "I'll remember that you prefer Python."}
            ],
            user_id=user_id,  # Use the same user_id from Part 1
            memory="User's favorite programming language is Python",
            metadata={
                "session_id": "demo_session",
                "agent_id": "demo_agent",
                "type": "preference"
            },
            tags=["programming", "preference", "python"]
        )
        
        # Add the memory - project already exists from Part 1
        await memory_client.add_items([initial_memory])
        print(f"✅ Added initial memory for user: {user_id}")
        print("   Memory: User's favorite programming language is Python")
        
        await asyncio.sleep(2)  # Wait for indexing

await setup_initial_memory()

Using existing project: demo_org_f888f411/demo_project_f888f411
✅ Added initial memory for user: demo_user_f888f411
   Memory: User's favorite programming language is Python


### 2.2) Verify LangChain Plugin and Run Agent

**Important:** The `react_agent` workflow requires the langchain plugin. When running `!nat run` from Jupyter, it may use system Python instead of the venv, causing the langchain plugin to not be found.

**Solution:** We use `python -m nat.cli.main run` instead of `!nat run` to ensure we use the venv's Python interpreter, which has access to the langchain plugin.

In [17]:
# Run the agent using the venv's Python to ensure langchain plugin is accessible
# Using sys.executable ensures we use the same Python as the notebook kernel (venv)
import sys

# Use python -m nat.cli.main instead of !nat to ensure correct Python environment
!{sys.executable} -m nat.cli.main run --config_file memmachine_agent_config.yml --input "What is my favorite programming language?"

2026-01-08 13:57:49 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'memmachine_agent_config.yml'

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 1
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-01-08 13:57:54 - INFO     - nat.agent.react_agent.agent:169 - 
------------------------------
[AGENT]
Agent input: What is my favorite programming language?
Agent's thoughts: 
Thought: I need to check if the user has mentioned their favorite programming language before.
Action: get_memory
Action Input: {"query": "favorite programming language", "top_k": 5, "user_id": "demo_user_f888f411"}

------------------------------
2026-01-08 13:57:55 - INFO     - memmachine.rest_client.memory:396 - Search completed for query: favorite programming language
2026-01-08 13:57:5

In [18]:
# Verify that the memmachine plugin is properly installed and registered
import sys
from importlib.metadata import entry_points

try:
    # Check if the entry point exists
    components = entry_points(group='nat.components')
    memmachine_entry = None
    for ep in components:
        if 'memmachine' in ep.name.lower():
            memmachine_entry = ep
            break
    
    if memmachine_entry:
        print(f"✅ Found memmachine entry point: {memmachine_entry.name} -> {memmachine_entry.module}")
        try:
            # Try to load the module to verify it works
            memmachine_entry.load()
            print("✅ Plugin module loaded successfully")
        except Exception as e:
            print(f"❌ Failed to load plugin module: {e}")
            print("\n💡 Try reinstalling the package in editable mode:")
            print("   uv pip install --force-reinstall -e ../../packages/memmachine")
    else:
        print("❌ memmachine entry point not found!")
        print("\n💡 The plugin may not be installed correctly.")
        print("   Try reinstalling in editable mode:")
        print("   uv pip install --force-reinstall -e ../../packages/memmachine")
        print("\n   Or if you're not in the repo directory:")
        print("   uv pip install --force-reinstall nvidia-nat-memmachine")
except Exception as e:
    print(f"❌ Error checking entry points: {e}")
    print("\n💡 Try reinstalling the package:")
    print("   uv pip install --force-reinstall -e ../../packages/memmachine")

✅ Found memmachine entry point: nat_memmachine -> nat.plugins.memmachine.register
✅ Plugin module loaded successfully


Great! The agent should have retrieved the memory about Python. Now let's test adding a new memory through the agent:

In [19]:
# Use Python's -m flag to ensure we use the venv's nat module
import sys
!{sys.executable} -m nat.cli.main run --config_file memmachine_agent_config.yml --input "I love reading science fiction novels, especially Dune."

2026-01-08 13:58:00 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'memmachine_agent_config.yml'

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 1
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-01-08 13:58:11 - INFO     - nat.agent.react_agent.agent:169 - 
------------------------------
[AGENT]
Agent input: I love reading science fiction novels, especially Dune.
Agent's thoughts: 
Thought: I need to check if the user has any existing preferences or facts about science fiction novels.

Action: get_memory
Action Input: {"query": "science fiction novels", "top_k": 5, "user_id": "demo_user_f888f411"}

------------------------------
2026-01-08 13:58:12 - INFO     - memmachine.rest_client.memory:396 - Search completed for query: science fiction novels
2026-01

Now let's verify the agent can recall this new memory:

In [20]:
# Use Python's -m flag to ensure we use the venv's nat module
import sys
!{sys.executable} -m nat.cli.main run --config_file memmachine_agent_config.yml --input "What books do I like to read?"

2026-01-08 13:58:41 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'memmachine_agent_config.yml'

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 2
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 1
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2026-01-08 13:58:45 - INFO     - nat.agent.react_agent.agent:169 - 
------------------------------
[AGENT]
Agent input: What books do I like to read?
Agent's thoughts: 
Thought: I need to check if there are any existing memories about the user's book preferences.

Action: get_memory
Action Input: {"query": "book preferences", "top_k": 5, "user_id": "demo_user_f888f411"}


------------------------------
2026-01-08 13:58:46 - INFO     - memmachine.rest_client.memory:396 - Search completed for query: book preferences
2026-01-08 13:58:46 - INFO     - nat.agent.base:221 - 

<a id="next-steps"></a>
## 3) Next Steps

Congratulations! You've successfully integrated MemMachine memory with NeMo Agent Toolkit. Here are some next steps to explore:

1. **Explore Advanced Memory Features**:
   - Use metadata and tags for better memory organization
   - Experiment with different memory types (episodic vs semantic)
   - Try memory deletion and cleanup strategies

2. **Integrate with Other Components**:
   - Combine memory with RAG (Retrieval Augmented Generation)
   - Use memory in multi-agent workflows
   - Add memory to custom tools and functions

3. **Production Considerations**:
   - Set up proper Neo4j database management
   - Configure memory retention policies
   - Implement memory search optimization
   - Add monitoring and observability

4. **Additional Resources**:
   - [MemMachine Documentation](https://docs.memmachine.ai/)
   - [NeMo Agent Toolkit Documentation](https://docs.nvidia.com/nemo/agent-toolkit/latest/)
   - [Memory Module Guide](https://docs.nvidia.com/nemo/agent-toolkit/latest/build-workflows/memory.html)
